# pymefpp_create_associations

This notebook is used to create the associations required to run mefpp


# Testing notebook

It works without any problem or warning

In [ ]:
import os
import sys
import glob
import logging

from pathlib import Path

# here we import the local mefpp_run library
sys.path.insert(0, os.getcwd())
import mefpp_associate_reduction as mar

import pycavi_tools
import pycavi_instrument

In [ ]:
# Reading configuration file
#
# Asume that all notebooks are in a directory where the data reduction will
# be carried out from. It will contain initially all python libraries and 
# a the config subdirectory.
#
# debug_mode: True
# log_level: info
# observatory: Paranal
# telescope: VST
# instrument: OmegaCam
# file_prefix: OMEGA
# raw_data_dir: /Users/fselman/Pictures/OCAM/CPPup/raw
# project_dir: /Users/fselman/Pictures/OCAM/CPPup2
# reduced_calib_dir: /Users/fselman/Pictures/OCAM/CPPup2/calib/
# reduced_science_dir: /Users/fselman/Pictures/OCAM/CPPup2/science/
# project_logs_dir: /Users/fselman/Pictures/OCAM/CPPup2/logs/
# sex_dir_name: Sextractor_dir
# scamp_dir_name: Scamp_dir
# swarp_dir_name: Swarp_dir
# cal_dir_name: Calibration_dir
#
config_dir = os.getcwd()+'/config/'
config = pycavi_tools.read_pycavi_config(config_dir)
raw_data_dir = config['Directories']['raw_data_dir']
project_dir = config['Directories']['project_dir']
static_cal_dir = config['Directories']['static_calib_dir']
reduced_cal_dir = config['Directories']['reduced_calib_dir']
reduced_science_dir = config['Directories']['project_dir']     # Note that for this notebook we set the reduced science dir to the root project dir.
log_dir = config['Directories']['project_logs_dir']

# Should get these from an instrument configuration file
instrument = pycavi_instrument.Instrument(config_dir, instrument='OCAM')
bpm_prefix = instrument.bpm_prefix
bias_prefix = instrument.bias_prefix
domeff_prefix = instrument.domeff_prefix
skyff_prefix = instrument.skyff_prefix
mff_prefix = instrument.mff_prefix


# we prepare the logs
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_dir+"/pymeffpp_associate.log"),
                              logging.StreamHandler()
                    ])

In [ ]:
# We identify only two types of partitions:
#    - BiasPartitions: only need OBstart time to separate them
#    - SciencePartitions: need filter and OBstart
#
bpm_list = glob.glob(static_cal_dir+'**/bpm.list', recursive=True)
logging.info('number of bpm partitions: {0}'.format(len(bpm_list)))
bpm_partitions = []
for bpm_file in bpm_list:
    bpm_partitions.append(mar.BiasPartition(bpm_file))
    
for part in bpm_partitions:
    logging.info(part.OBstart)

In [ ]:
bias_list = glob.glob(reduced_cal_dir+'**/bias.list', recursive=True)
print('number of bias partitions: ', len(bias_list))
bias_partitions = []
for bias_file in bias_list:
    bias_partitions.append(mar.BiasPartition(bias_file))
    
for part in bias_partitions:
    print(part.OBstart)

In [ ]:
dome_list = glob.glob(reduced_cal_dir+'**/dome.list', recursive=True)
print('number of dome partitions: ', len(dome_list))
dome_partitions = []
for dome_file in dome_list:
    dome_partitions.append(mar.SciencePartition(dome_file))
    
for part in dome_partitions:
    print(part.OBstart, part.filter)

In [ ]:
sky_list = glob.glob(reduced_cal_dir+'**/sky.list', recursive=True)
print('number of sky partitions: ', len(sky_list))
sky_partitions = []
for sky_file in sky_list:
    sky_partitions.append(mar.SciencePartition(sky_file))
    
for part in sky_partitions:
    print(part.OBstart, part.filter)

In [ ]:
science_list = glob.glob(reduced_science_dir+'**/science.list', recursive=True)
print('number of science partitions: ', len(science_list))

sci_partitions = []
for science_file in science_list:
    sci_partitions.append(mar.SciencePartition(science_file))
    
for part in sci_partitions:
    print(part.OBstart, part.filter)



In [ ]:
exposurePartition = mar.PartitionsTree(raw_data_dir, static_cal_dir, reduced_cal_dir, reduced_science_dir, log_dir, instrument)
bpm_partitions = exposurePartition.bpm_partitions
bias_partitions = exposurePartition.bias_partitions
dome_partitions = exposurePartition.dome_partitions
sky_partitions = exposurePartition.sky_partitions
sci_partitions = exposurePartition.sci_partitions

In [ ]:
# We associate a bpm partition to each science frame and create symbolic link in
# corresponding directory
for sci_part in sci_partitions:
    sci_part.bpmPartition = bpm_partitions[0]
    for i in range(1,instrument.next_valid+1):
        bpm_file = (sci_part.bpmPartition).base_path+'/'+bpm_prefix+'_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, sci_part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((sci_part.bpmPartition).base_path+'/toreduce_sci').touch()
        print(bpm_file, sci_part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))


In [ ]:
# we associate a bias and master flat field partitions to each science partition

for sci_part in sci_partitions:
    # here we look for the closest in time bias partition
    sci_date = sci_part.OBstart

    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = mar.timeDistance(sci_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            sci_part.biasPartition = bias_part
    
    # we find the sky closest in time and we link the mff to a file that
    # will be present there once it is created
    #
    skyBestDeltaT = 3600.*24.*30.0
    for sky_part in sky_partitions:
        if sci_part.filter != sky_part.filter:
            continue
        sky_date = sky_part.OBstart
        deltaT = mar.timeDistance(sci_date, sky_date)

        if deltaT < skyBestDeltaT:
            skyBestDeltaT = deltaT
            sci_part.skyPartition = sky_part
            sci_part.mffPartition = sky_part

    
    print('science: %s %s'%(sci_part.filter, sci_part.OBstart,)+\
            '     bias: %s (%3.1f)'%(sci_part.biasPartition.OBstart,biasBestDeltaT/3600.0/24.,)+\
            ' sky: %s %s (%3.1f)'%(sci_part.skyPartition.filter, sci_part.skyPartition.OBstart, skyBestDeltaT/3600.0/24.,))

# at this point we have the biases and mff for each science frame.
# Now we create virtual calibration products as a symbolic link.
#
for sci_part in sci_partitions:
    mbias = (sci_part.biasPartition).base_path+'/'+bias_prefix+'_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, sci_part.base_path+'/'+bias_prefix+'_1x1.fits')
        except:
            pass
        Path((sci_part.biasPartition).base_path+'/toreduce_sci').touch()
        print(mbias)

    mff = (sci_part.mffPartition).base_path+'/'+mff_prefix+'_'+sci_part.filter+'_1x1.fits'
    if not Path(mff).is_file():
        try:
            os.symlink(mff, sci_part.base_path+'/'+mff_prefix+'_'+sci_part.filter+'_1x1.fits')
        except:
            pass
        Path((sci_part.mffPartition).base_path+'/toreduce_mff').touch()


In [ ]:
#We associate a bpm partition to each sky frame partition
for sky_part in sky_partitions:
    sky_part.bpmPartition = bpm_partitions[0]
    for i in range(1,instrument.next_valid+1):
        bpm_file = (sky_part.bpmPartition).base_path+'/'+bpm_prefix+'_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, sky_part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((sky_part.bpmPartition).base_path+'/toreduce_sky').touch()
        print(bpm_file, sky_part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))

In [ ]:
# here we associate dome and bias to each sky partition
for sky_part in sky_partitions:
    sky_date = sky_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = mar.timeDistance(sky_date, bias_date)
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            sky_part.biasPartition = bias_part
    
    # we find the dome closest in time and we link the domeff to a file that
    # will be present there once it is created
    #
    domeBestDeltaT = 3600.*24.*30.0
    for dome_part in dome_partitions:
        if sky_part.filter == dome_part.filter:
            dome_date = dome_part.OBstart
            deltaT = mar.timeDistance(sky_date, dome_date)
            #print(sci_date, bias_date, deltaT)
            if deltaT < domeBestDeltaT:
                domeBestDeltaT = deltaT
                sky_part.domePartition = dome_part
        else:
            pass

    
    print('sky: %s %s'%(sky_part.filter, sky_part.OBstart,)+\
            '     bias: %s (%4.2f)'%(sky_part.biasPartition.OBstart,biasBestDeltaT/3600.0/24.,)+\
            ' dome: %s %s (%4.2f)'%(sky_part.domePartition.filter, sky_part.domePartition.OBstart, domeBestDeltaT/3600.0/24.,))
    
# at this point we have the biases and domes for each sky ff frame. Now we link
for sky_part in sky_partitions:
    mbias = (sky_part.biasPartition).base_path+'/'+bias_prefix+'_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, sky_part.base_path+'/'+bias_prefix+'_1x1.fits')
        except:
            pass
        Path((sky_part.biasPartition).base_path+'/toreduce_sky').touch()
        
    domeff = (sky_part.domePartition).base_path+'/'+domeff_prefix+'_'+sky_part.filter+'_1x1.fits'
    if not Path(domeff).is_file():
        try:
            os.symlink(domeff, sky_part.base_path+'/'+domeff_prefix+'_'+sky_part.filter+'_1x1.fits')
        except:
            pass
        Path((sky_part.domePartition).base_path+'/toreduce_sky').touch()


In [ ]:
#We associate a bpm partition to each dome ff frame
for dome_part in dome_partitions:
    dome_part.bpmPartition = bpm_partitions[0]
    for i in range(1,instrument.next_valid+1):
        bpm_file = (dome_part.bpmPartition).base_path+'/'+bpm_prefix+'_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, dome_part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((dome_part.bpmPartition).base_path+'/toreduce_dome').touch()

    dome_date = dome_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = mar.timeDistance(dome_date, bias_date)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            dome_part.biasPartition = bias_part        
    
    mbias = (dome_part.biasPartition).base_path+'/'+bias_prefix+'_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, dome_part.base_path+'/'+bias_prefix+'_1x1.fits')
        except:
            pass
        Path((dome_part.biasPartition).base_path+'/toreduce_dome').touch()

In [ ]:
# here we associate biases to each dome partition
for dome_part in dome_partitions:
    dome_date = dome_part.OBstart
    
    # we find the bias closest in time
    biasBestDeltaT = 3600.*24.*10.0
    for bias_part in bias_partitions:
        bias_date = bias_part.OBstart
        deltaT = mar.timeDistance(dome_date, bias_date)
        
        #print(sci_date, bias_date, deltaT)
        if deltaT < biasBestDeltaT:
            biasBestDeltaT = deltaT
            dome_part.biasPartition = bias_part

    
    print('dome: %s %s'%(dome_part.filter, dome_part.OBstart,)+\
            '     bias: %s (%5.3f)'%((dome_part.biasPartition).OBstart, biasBestDeltaT/3600.0/24.,))
    
# at this point we have the biases  for each dome ff frame. Now we link
for dome_part in dome_partitions:
    mbias = (dome_part.biasPartition).base_path+'/'+bias_prefix+'_1x1.fits'
    if not Path(mbias).is_file():
        try:
            os.symlink(mbias, dome_part.base_path+'/'+bias_prefix+'_1x1.fits')
        except:
            pass
        Path((dome_part.biasPartition).base_path+'/toreduce_dome').touch()
        

In [ ]:
# Finally associate a bpm partition to each bias frame
for bias_part in bias_partitions:
    bias_part.bpmPartition = bpm_partitions[0]
    for i in range(1,instrument.next_valid+1):
        bpm_file = (bias_part.bpmPartition).base_path+'/'+bpm_prefix+'_%02d.fits'%(i,)
        if not Path(bpm_file).is_file():
            try:
                os.symlink(bpm_file, bias_part.base_path+'/'+bpm_prefix+'_%02d.fits'%(i,))
                pass
            except:
                pass
            Path((bias_part.bpmPartition).base_path+'/toreduce_bias').touch()
